# pypasi runtime and memory against a single classifier

Answers Reviewer 1, comment 2 of the SoftwareX submission. Every step a user runs is timed on
**1,000 spectra** and compared with the same base estimator, PLS-DA(10), fitted once on the
whole spectrum.

- Timings are wall-clock medians over `REPEATS` runs.
- Peak memory is the largest Python allocation during the step (`tracemalloc`). It is measured
  in a separate pass, so tracing does not inflate the timings.
- Set `DATA_DIR = None` to run on synthetic spectra of the same shape; no download is needed.

In [1]:
from pathlib import Path

DATA_DIR = Path("Bacteria")   # Ho et al. .npy files; None for synthetic spectra
N        = 1000               # spectra fitted, and spectra predicted
N_BANDS  = 7
CHANNELS = 1000               # synthetic data only
REPEATS  = 5
SEED     = 0

In [2]:
import os, platform, time, tracemalloc
import numpy as np
import pandas as pd
import sklearn, scipy
import pypasi
from pypasi import (BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage,
                    audit_external)

def _cpu():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

print(f"pypasi {pypasi.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"scikit-learn {sklearn.__version__} | Python {platform.python_version()}")
print(f"{_cpu()} | {os.cpu_count()} logical CPUs | {platform.system()}")

pypasi 0.5.5 | numpy 2.5.3 | scipy 1.18.1 | scikit-learn 1.9.1 | Python 3.13.17
Intel(R) Xeon(R) Processor @ 2.10GHz | 2 logical CPUs | Linux


## Data

The paper's representation: per-spectrum min–max, then SNV.

In [3]:
def snv(X):
    X = np.asarray(X, float)
    return (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)

if DATA_DIR is not None:
    from pypasi.datasets import load_bacteria
    d = load_bacteria(DATA_DIR, normalize=False)
    X = d.X_dev
    X = (X - X.min(1, keepdims=True)) / (X.max(1, keepdims=True) - X.min(1, keepdims=True))
    X, y, axis, source = snv(X), d.y_dev, d.axis, "Ho et al. 2018 cohort"
else:
    from pypasi.datasets import make_conflict_signals
    X, y, axis, _ = make_conflict_signals(n_samples=8000, n_classes=5, n_points=CHANNELS,
                                          random_state=SEED)
    X, source = snv(X), "synthetic (make_conflict_signals)"

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
fit, cal, new = idx[:N], idx[N:N + N // 2], idx[2 * N:3 * N]
ref = idx[3 * N:3 * N + 2000]
Xf, yf, Xc, yc, Xn, yn = X[fit], y[fit], X[cal], y[cal], X[new], y[new]
batches = [X[ref[i::20]] for i in range(20)]
bands = BandSet.equal_width(axis, N_BANDS, lo=max(400.0, float(axis.min())), hi=float(axis.max()))
print(f"{source}: {X.shape[1]} channels | fit {len(Xf)} | calibration {len(Xc)} | "
      f"predict {len(Xn)} | {N_BANDS} bands")

Ho et al. 2018 cohort: 1000 channels | fit 1000 | calibration 500 | predict 1000 | 7 bands


## The steps

Each step is a function, so the same code is used for timing and for memory.

In [4]:
S = {}

def base_fit():       S["base"] = PLSDA(n_components=10).fit(Xf, yf)
def base_predict():   S["base"].predict(Xn)
def bnc_fit():
    S["bnc"] = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=10), topology="complete", regime="H1",
        calibration="auto", calibration_data=(Xc, yc), random_state=SEED).fit(Xf, yf)
def bnc_predict():    S["bnc"].predict(Xn)
def bnc_audit():      S["bnc"].audit(Xn)
def ext_fit():
    S["ext"] = audit_external(S["base"], Xf, yf, bands=bands, topology="complete", regime="H1",
                              base_estimator=PLSDA(n_components=10), calibration="auto",
                              calibration_data=(Xc, yc), random_state=SEED)
def ext_explain():    S["ext"].explain(Xn)
def triage_fit():     S["tri"] = Triage(conflict_score="dg", n_components=10).fit(S["bnc"], Xc, yc)
def triage_assess():  S["tri"].assess(Xn, yn)
def profile_fit():    S["prof"] = ControlProfile.fit(S["bnc"], batches=batches, random_state=SEED)
def profile_check():  S["prof"].check(Xn[:100])

STEPS = [
    ("baseline", "full-spectrum PLS-DA(10): fit, 1,000", base_fit),
    ("baseline", "full-spectrum PLS-DA(10): predict, 1,000", base_predict),
    ("pypasi", "BandNegotiationClassifier.fit, 1,000", bnc_fit),
    ("pypasi", "BandNegotiationClassifier.predict, 1,000", bnc_predict),
    ("pypasi", "BandNegotiationClassifier.audit, 1,000 (trajectories kept)", bnc_audit),
    ("pypasi", "audit_external: fit around the baseline", ext_fit),
    ("pypasi", "ExternalAudit.explain, 1,000", ext_explain),
    ("pypasi", "Triage.fit", triage_fit),
    ("pypasi", "Triage.assess, 1,000", triage_assess),
    ("pypasi", "ControlProfile.fit, 20 batches of 100", profile_fit),
    ("pypasi", "ControlProfile.check, 1 batch of 100", profile_check),
]

## Timing, then memory

In [5]:
import warnings
warnings.filterwarnings("ignore")      # e.g. the random-split warning of ControlProfile

times = {name: [] for _, name, _ in STEPS}
for _ in range(REPEATS):
    for _, name, f in STEPS:
        t0 = time.perf_counter(); f(); times[name].append(time.perf_counter() - t0)

peak = {}
for _, name, f in STEPS:
    tracemalloc.start(); f(); peak[name] = tracemalloc.get_traced_memory()[1] / 2**20
    tracemalloc.stop()

tab = pd.DataFrame([{"group": g, "step": n,
                     "median_s": np.median(times[n]), "min_s": np.min(times[n]),
                     "max_s": np.max(times[n]), "peak_MB": peak[n]} for g, n, _ in STEPS])
tab.to_csv("bench_runtime.csv", index=False)
tab.round(3)

,group,step,median_s,min_s,max_s,peak_MB
0,baseline,"full-spectrum PLS-DA(10): fit, 1,000",0.247,0.221,0.297,23.469
1,baseline,"full-spectrum PLS-DA(10): predict, 1,000",0.009,0.008,0.012,15.375
2,pypasi,"BandNegotiationClassifier.fit, 1,000",0.177,0.173,0.200,6.326
3,pypasi,"BandNegotiationClassifier.predict, 1,000",0.272,0.262,0.323,10.166
4,pypasi,"BandNegotiationClassifier.audit, 1,000 (trajec...",0.284,0.282,0.298,25.342
5,pypasi,audit_external: fit around the baseline,0.741,0.613,0.799,39.670
6,pypasi,"ExternalAudit.explain, 1,000",0.415,0.344,0.453,43.402
7,pypasi,Triage.fit,0.424,0.375,0.545,20.192
8,pypasi,"Triage.assess, 1,000",0.342,0.292,0.354,43.249
9,pypasi,"ControlProfile.fit, 20 batches of 100",1.890,1.865,2.019,4.227


## Summary

In [6]:
t = tab.set_index("step")["median_s"]
fit_b, fit_p = t.iloc[0], t.iloc[2]
pred_b, pred_p = t.iloc[1], t.iloc[3]
print(f"fit     : pypasi {fit_p:.2f} s  vs  full-spectrum PLS-DA {fit_b:.2f} s")
print(f"predict : pypasi {pred_p:.2f} s  vs  full-spectrum PLS-DA {pred_b:.3f} s "
      f"-> the negotiation adds {1000 * (pred_p - pred_b) / N:.2f} ms per spectrum")
print(f"largest peak allocation of any step: {tab['peak_MB'].max():.0f} MB")

fit     : pypasi 0.18 s  vs  full-spectrum PLS-DA 0.25 s
predict : pypasi 0.27 s  vs  full-spectrum PLS-DA 0.009 s -> the negotiation adds 0.26 ms per spectrum
largest peak allocation of any step: 43 MB


## Cost against the number of bands and the topology

The audit of 1,000 spectra, median of `REPEATS` runs. The complete graph has k(k − 1)/2 edges, and kNN (k = 2) about 2k, so the gap opens as bands are added.

In [7]:
scal = []
for k in (7, 14, 28):
    bk = BandSet.equal_width(axis, k, lo=max(400.0, float(axis.min())), hi=float(axis.max()))
    for topo in ("complete", "knn"):
        m = BandNegotiationClassifier(bands=bk, base_estimator=PLSDA(n_components=10), topology=topo,
                                      regime="H1", calibration="auto", calibration_data=(Xc, yc),
                                      random_state=SEED).fit(Xf, yf)
        ts = []
        for _ in range(REPEATS):
            t0 = time.perf_counter(); m.audit(Xn); ts.append(time.perf_counter() - t0)
        scal.append({"bands": k, "topology": topo, "audit_1000_s": float(np.median(ts))})
scal = pd.DataFrame(scal).pivot(index="bands", columns="topology", values="audit_1000_s")
scal.to_csv("bench_runtime_bands.csv")
scal.round(3)

topology,complete,knn
bands,,
7,0.269,0.225
14,0.948,0.562
28,5.070,1.289
